### Problem 001: Kth Largest Element in a Stream (LeetCode 703)

### Problem Definition and Constraints
Design a class to find the $k$-th largest integer in a stream of values, including duplicates. The stream is not necessarily sorted.
Implement the `KthLargest` class:
* `KthLargest(int k, int[] nums)` Initializes the object with the integer `k` and the stream of integers `nums`.
* `int add(int val)` Appends the integer `val` to the stream and returns the $k$-th largest element in the stream.

* Constraints:
  * 1 <= k <= 10^4
  * 0 <= nums.length <= 10^4
  * -10^4 <= nums[i], val <= 10^4
  * There will always be at least `k` integers in the stream when you search for the $k$-th integer.

### Brute Force Approach
Every time a new number is added via `add(val)`, we append it to a standard array and then sort the entire array in descending order. Once sorted, we simply return the element at index `k - 1`.
* Time Complexity: $O(m \cdot n \log n)$ — Where $m$ is the number of `add` calls and $n$ is the total elements. Sorting the entire list on every single insertion is extremely slow.
* Space Complexity: $O(n)$ — We store every single number that ever gets added to the stream.

### Optimized Approach (Min-Heap)
To achieve $O(\log k)$ insertions, we use a Min-Heap. By strictly limiting the size of the heap to exactly `k` elements, the heap will naturally hold only the `k` largest numbers seen so far. Because it is a *Min*-Heap, the smallest number out of those `k` largest numbers will always sit at the very top (index 0). When `add()` is called, we push the new value onto the heap. If the heap's size exceeds `k`, we immediately pop the top element (which removes the smallest number, kicking out the "poorest" VIP). We then return the new top element.
* Time Complexity: $O(n \log k)$ for initialization, and $O(\log k)$ for each `add()` call. Pushing and popping from a heap of size $k$ takes logarithmic time relative to $k$.
* Space Complexity: $O(k)$ — We permanently discard any numbers that aren't in the top `k`, so our memory footprint never grows beyond size `k`.

In [ ]:
import heapq
from typing import List

class KthLargest:

    def __init__(self, k: int, nums: List[int]):
        # Store k so we know the maximum size of our VIP club
        self.k = k
        self.minHeap = nums
        
        # heapq.heapify transforms a standard list into a Min-Heap in-place in O(n) time
        heapq.heapify(self.minHeap)
        
        # If the starting array has more than k elements, pop the smallest ones 
        # until we are down to exactly k elements.
        while len(self.minHeap) > self.k:
            heapq.heappop(self.minHeap)

    def add(self, val: int) -> int:
        # 1. Push the new person into the club
        heapq.heappush(self.minHeap, val)
        
        # 2. If the club has more than k people, kick out the poorest one
        if len(self.minHeap) > self.k:
            heapq.heappop(self.minHeap)
            
        # 3. The poorest person in the VIP club (the root of the min-heap) 
        # is the k-th largest element overall.
        return self.minHeap[0]

# Your KthLargest object will be instantiated and called as such:
# obj = KthLargest(k, nums)
# param_1 = obj.add(val)


### Problem 002: Last Stone Weight (LeetCode 1046)

### Problem Definition and Constraints
You are given an array of integers `stones` where `stones[i]` represents the weight of the $i$-th stone.
We repeatedly choose the two heaviest stones and smash them together. 
* If `x == y`, both stones are destroyed.
* If `x < y`, the stone of weight `x` is destroyed, and the stone of weight `y` has a new weight of `y - x`.
Return the weight of the last remaining stone or return 0 if none remain.

* Constraints:
  * 1 <= stones.length <= 30
  * 1 <= stones[i] <= 1000

### Examples
* **Example 1:**
  * Input: `stones = [2,7,4,1,8,1]`
  * Output: `1`
  * Explanation: 
    * Smash 8 and 7 -> 1. Array becomes `[2,4,1,1,1]`
    * Smash 4 and 2 -> 2. Array becomes `[2,1,1,1]`
    * Smash 2 and 1 -> 1. Array becomes `[1,1,1]`
    * Smash 1 and 1 -> 0. Array becomes `[1]`
    * Last stone is 1.

### Brute Force Approach
The naive approach is to use a standard array. Inside a `while` loop, we sort the entire array in descending order, pop the first two elements, calculate their difference, and append the result back to the array. We repeat this until the array length is 1 or 0.
* Time Complexity: $O(n^2 \log n)$ — We have to re-sort the entire array of size $n$ every single time we do a smash operation (which happens roughly $n$ times).
* Space Complexity: $O(1)$ or $O(n)$ depending on if the sorting is done in-place.

### Optimized Approach (Max-Heap Simulation)
To avoid resorting the whole array every time, we use a Max-Heap. Because Python's `heapq` library only supports Min-Heaps, we first negate all values in the array (e.g., `5` becomes `-5`). The "heaviest" stone becomes the "smallest" negative number, naturally bubbling to the top of the Min-Heap.
While there is more than 1 stone in the heap, we pop the top two elements (multiplying by -1 to get their true positive weights back). If the first is heavier than the second, we calculate `first - second`, negate it, and push it back into the heap. If they are equal, we push nothing. We return the final stone (made positive again) or `0` if the heap is empty.
* Time Complexity: $O(n \log n)$ — `heapify` takes $O(n)$. We then do at most $n$ smashes. Each smash requires two `heappop` and one `heappush` operations, taking $O(\log n)$ time each.
* Space Complexity: $O(n)$ — We store the negated weights in a heap structure of size $n$.


In [ ]:
import heapq
from typing import List

class Solution:
    def lastStoneWeight(self, stones: List[int]) -> int:
        
        # 1. Transform into a Max-Heap using the "Negative Trick"
        # We multiply every stone by -1 so the heaviest stones become the smallest numbers
        max_heap = [-s for s in stones]
        heapq.heapify(max_heap)
        
        # 2. Simulate the gladiator arena
        # We need at least 2 stones to have a fight
        while len(max_heap) > 1:
            
            # Pop the two "smallest" (most negative) numbers and make them positive again
            first = -heapq.heappop(max_heap)   # The absolute heaviest stone
            second = -heapq.heappop(max_heap)  # The second heaviest stone
            
            # If the first is bigger, there is a leftover piece.
            if first > second:
                leftover = first - second
                # Push the leftover back into the ring (remember to make it negative!)
                heapq.heappush(max_heap, -leftover)
                
            # If first == second, both are destroyed. We do nothing and loop again.
            
        # 3. Check the aftermath
        # If there is a survivor, make it positive and return it.
        if max_heap:
            return -max_heap[0]
            
        # If they completely wiped each other out, return 0.
        return 0